In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent 
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
load_dotenv()

True

In [8]:
model = ChatOpenAI(model="gpt-4o-mini")


In [9]:
prompt = """
Understand the coding request:

For any request, identify the following :

- task type
- objective
- files involved 

Return the response in the following JSON format:
{
    "task_type": "type of task",
    "objective": "objective of the task",
    "files_involved": "list of files involved"
}

"""

In [10]:
request = "Identify the bug in the auth.py file where the exchange token is not getting generated"


In [11]:
response=model.invoke(prompt + request)
print(response.content)

```json
{
    "task_type": "bug identification",
    "objective": "identify the bug in the auth.py file where the exchange token is not getting generated",
    "files_involved": ["auth.py"]
}
```


In [12]:
# PURPOSE OF THIS CLASS:
# It is a FORM the model must fill in. Instead of asking for JSON in the prompt
# and hoping, we hand the model this shape and it MUST answer in it.
#
# EVERY LINE BELOW FOLLOWS THE SAME PATTERN:
#
#     field_name : WhatTypeItMustBe = Field(description="what to put here")
#     ^^^^^^^^^^   ^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#     the name     the RULE           the INSTRUCTION the model reads
#
#   :  the colon introduces a TYPE ANNOTATION - the rule for what may go here
#   =  the equals attaches EXTRA INFO about the field (here, a description)


# BaseModel is pydantic's base class. Inheriting from it is what gives this
# class its two powers: it can VALIDATE data, and it can describe itself as a
# JSON schema (which is what actually gets sent to the model).
class CodingRequest(BaseModel):
    """Structured representation of  the coding request"""
    # ^ this docstring is also sent to the model, as the schema's description.

    # -------------------------------------------------------------------------
    # Literal means: "the value must be EXACTLY one of these four strings."
    # Not "a string" - one of THESE. Anything else fails validation.
    #
    # This is the whole reason to use it. In the plain-prompt cell above, the
    # model invented "bug identification" when you wanted "debug". Literal makes
    # that impossible, so `if task_type == "debug"` is now safe to write.
    # -------------------------------------------------------------------------
    task_type: Literal[
        "implement",
        "debug",
        "review",
        "explain",
        
    ] = Field(description="Type of task to be performed")
    #   ^^^^^ Field() is NOT a default value. It is a box holding EXTRA
    #   information about this field. The description inside it is copied into
    #   the JSON schema, so the model literally reads this sentence to decide
    #   what belongs here. Without Field() the model would only see the name
    #   "task_type" and have to guess.

    # A plain string. No restriction on the wording, only on the type.
    objective: str = Field(description="A concise description of what needs to be achieved")

    # bool = True or False only. The model cannot answer "maybe" or "yes".
    need_code_changes: bool = Field(
        description="Whether fulfilling this task required modifying the source code"
    )

    # list[str] = a list, and every item inside it must be a string.
    # Guaranteed to be a real Python list - so you can safely loop over it
    # without checking whether the model returned a comma-separated string.
    target_files: list[str] = Field(
        description="List of files that are relevant to the task, e.g. ['auth.py', 'models.py']"
    )
    # The "e.g." in that description is doing real work: it shows the model the
    # exact format you want, which is more reliable than describing it.


# HOW YOU USE IT (not done in this notebook yet):
#
#     structured_model = model.with_structured_output(CodingRequest)
#     result = structured_model.invoke(request)
#
#     result.task_type      -> 'debug'        guaranteed one of the four
#     result.target_files   -> ['auth.py']    guaranteed a list of strings
#
# You get back a CodingRequest OBJECT, not a string. No ```json fence to strip,
# no json.loads, no checking whether a key exists.




In [ ]:
#modified model  object 
#first step is the schema conversion 
#output when we call the model  will  be an  object of  CodingRequest class , it sends  the schema to the model  so  the model  knows in what format to  respond 
# langchain  does the heavy lifting and decides from the models schema how to  build the answer 
structured_output_model = model.with_structured_output(CodingRequest)

In [ ]:
result = structured_output_model.invoke("""
    Fix the login bug in auth.py. Expired sessions currently produce HTTP 500.
""")

In [15]:
print(type(result))

<class '__main__.CodingRequest'>


In [ ]:
CodingRequest.model_json_schema() #produces  a json  string of pydantic , OpenAI  does not understand this 

{'description': 'Structured representation of  the coding request',
 'properties': {'task_type': {'description': 'Type of task to be performed',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A concise description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'Whether fulfilling this task required modifying the source code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'description': "List of files that are relevant to the task, e.g. ['auth.py', 'models.py']",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes', 'target_files'],
 'title': 'CodingRequest',
 'type': 'object'}

In [17]:
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(CodingRequest),
    system_prompt=prompt
)

In [18]:

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Identify the bug in the auth.py file where the exchange token is not getting generated"}
    ]
})
print(type(result["structured_response"]))
print(result["structured_response"])

<class '__main__.CodingRequest'>
task_type='debug' objective='Identify the bug in the auth.py file where the exchange token is not getting generated' need_code_changes=False target_files=['auth.py']


In [19]:
for message in result["messages"]:
    print(message.pretty_print())
    print("===================\n\n")


================================ Human Message =================================

Identify the bug in the auth.py file where the exchange token is not getting generated
None


================================== Ai Message ==================================
Tool Calls:
  CodingRequest (call_vO40DDXrCQy7fjQIwd6y44jw)
 Call ID: call_vO40DDXrCQy7fjQIwd6y44jw
  Args:
    task_type: debug
    objective: Identify the bug in the auth.py file where the exchange token is not getting generated
    need_code_changes: False
    target_files: ['auth.py']
None


================================= Tool Message =================================
Name: CodingRequest

Returning structured response: task_type='debug' objective='Identify the bug in the auth.py file where the exchange token is not getting generated' need_code_changes=False target_files=['auth.py']
None


